# Data Exploration

In [13]:
import sqlite3
import pandas as pd

## 1. Objective

The goal of this project is to develop a **predictive model capable of estimating the total sales value for the next six weeks for each store in a pharmacy chain**.

The forecast will be used to support strategic decisions, allowing the identification of stores with greater potential for return and assisting in planning investments in **renovations, expansion, and marketing campaigns**.

For each store and reference date, the model should estimate the **total sales value in the following 42 days**. Thus, the target is calculated as the sum of daily sales over the six weeks following the reference date.

For building the model, different sources of information about the stores are available:

* **Daily sales history**, containing information about sales, number of customers, store open status, promotions, and holidays;
* **Store characteristics**, such as type, assortment, and distance to competitors;
* **Promotion information**, including participation in promotional programs and promotional periods;
* **Temporal information**, used to identify seasonality patterns, trends, and sales behavior over time.

The solution should consider aspects such as:

* data quality and cleaning;
* variable creation and selection;
* construction of a **Feature Store by store and reference date**;
* creation of historical, temporal, seasonality, and trend variables;
* proper separation between training, validation, and test sets, respecting the temporal order of the data;
* evaluation of model performance;
* interpretation of results;
* comparison between the model predictions and different baseline approaches.

The model’s final output should contain **the predicted sales value for the next six weeks for each store**, representing the estimated sum of sales during the 42 days following the reference date.

> **Note:** the forecasting horizon used in this project is **42 days (six weeks)**. The model is not intended to forecast daily sales individually, but rather the **total expected sales value for each store over the following six weeks**.


## Database

The project provides two databases containing historical sales information and store characteristics for a pharmacy chain. These databases represent the daily activity of the stores as well as their main structural and promotional characteristics.

The tables are mainly related by:

* **Store**: uniquely identifies each store;
* **Date**: represents the reference date of the daily information for each store.

### Available Databases

| Database | Description                                                                                  | Granularity  |
|----------|----------------------------------------------------------------------------------------------|--------------|
| `sales`  | Daily information about sales, customers, store openings, promotions, and holidays.          | Store × day  |
| `store`  | Structural characteristics and information about competitors and promotional programs.        | Store        |


### Role of Each Database

The **sales** database contains the daily history of each store, including the number of sales, number of customers, whether the store was open, promotions, and holidays. This database serves as the main source for the creation of historical, temporal, seasonality, and trend variables.

The **store** database contains more structural information about each store, such as type, product assortment, distance to competitors, and participation in promotional programs. This information is linked to the `sales` database through the store identifier.

By combining both databases, a **Feature Store by store and reference date** is built, providing the necessary information for the predictive model.


## Sales

The **sales** database contains the daily historical information of the stores of a pharmacy chain. Each record represents a certain store on a specific date, enabling the monitoring of sales behavior, customer flow, store opening, promotions, and holidays over time.

The database has **1,017,209 records and 9 variables**.

During the initial data exploration, **no missing values** were identified in any of the variables. Particular attention should be paid to the `Date` variable, which is initially of type `object` and will need to be converted to date format to allow for the creation of temporal variables and historical windows used in the project.

Data Dictionary

| Variable        | Description                                         | Type    | Notes                                                                                      |
| --------------- | --------------------------------------------------- | ------- | ------------------------------------------------------------------------------------------ |
| `Store`         | Store identifier                                    | Integer | Used as a key to join with the `store` table                                               |
| `DayOfWeek`     | Day of the week                                     | Integer | Represents the day of the week of the observation                                          |
| `Date`          | Observation date                                    | Date    | Initially stored as `object`; used in the creation of temporal variables                   |
| `Sales`         | Amount of sales on the day                          | Integer | Main historical variable used for prediction                                               |
| `Customers`     | Number of customers served on the day               | Integer | Used to create variables related to customer flow                                          |
| `Open`          | Store open indicator                                | Integer | 1 = open; 0 = closed                                                                      |
| `Promo`         | Promotion indicator                                 | Integer | 1 = promotion; 0 = no promotion                                                           |
| `StateHoliday`  | State holiday indicator                             | Text    | Identifies the occurrence of a state holiday                                               |
| `SchoolHoliday` | School holiday indicator                            | Integer | 1 = school holiday; 0 = no school holiday                                                 |

Observed Points

* The database contains **1,017,209 rows and 9 columns**, with daily granularity per store.
* **No missing values** were identified in the variables.
* The `Date` variable is initially of type `object` and should be converted to `datetime` during data processing.
* `Sales` represents daily sales volume and will be used for constructing historical variables as well as for defining the **prediction target**.
* `Customers` represents the daily customer flow and will be used to create behavioral variables.
* `Open`, `Promo`, `StateHoliday`, and `SchoolHoliday` represent operational conditions and events that may influence sales behavior.
* `Store` will be used as the key to join the `sales` database with the structural characteristics present in the `store` table.
* The `Date` variable will be used as a reference for building historical windows, trend, and seasonality variables.
* To avoid **data leakage**, the variables used as model input must only consider information available up to the respective reference date.


In [ ]:
# Connect to the database
conn = sqlite3.connect('../../../data/database.db')

# List available tables in the database
tables = pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table';", conn)
print("Available tables in the database:", tables['name'].tolist())

# Load the sales table
df_sales = pd.read_sql_query("SELECT * FROM sales", conn)

conn.close()

Tabelas disponíveis no banco de dados: ['store', 'sales']


In [ ]:
# Display the first rows to check the structure of the sales DataFrame
df_sales.head()

,Store,DayOfWeek,Date,Sales,Customers,Open,Promo,StateHoliday,SchoolHoliday
0,1,5,2015-07-31,5263,555,1,1,0,1
1,2,5,2015-07-31,6064,625,1,1,0,1
2,3,5,2015-07-31,8314,821,1,1,0,1
3,4,5,2015-07-31,13995,1498,1,1,0,1
4,5,5,2015-07-31,4822,559,1,1,0,1


In [ ]:
# Display the number of rows and columns in the sales DataFrame
print(f"Number of rows: {df_sales.shape[0]}")
print(f"Number of columns: {df_sales.shape[1]}")

Número de linhas: 1017209
Número de colunas: 9


In [ ]:
# Check the data types of the sales table variables
df_sales.dtypes

Store             int64
DayOfWeek         int64
Date             object
Sales             int64
Customers         int64
Open              int64
Promo             int64
StateHoliday     object
SchoolHoliday     int64
dtype: object

In [ ]:
# Checking the unique values in the StateHoliday column
print("Distinct values of StateHoliday:", df_sales['StateHoliday'].unique())

Valores distintos de StateHoliday: ['0' 'a' 'b' 'c']


In [ ]:
# Calculate the percentage of missing values per column
missing_percent = df_sales.isnull().mean() * 100

print("Percentage of missing values per column:")
print(missing_percent)

Porcentagem de valores ausentes por coluna:
Store            0.0
DayOfWeek        0.0
Date             0.0
Sales            0.0
Customers        0.0
Open             0.0
Promo            0.0
StateHoliday     0.0
SchoolHoliday    0.0
dtype: float64


## Store

The **store** table contains structural and commercial characteristics of the pharmacy chain's stores. Each record represents an individual store, identified by the variable `Store`, and includes information related to store type, product assortment, competition, and participation in promotional programs.

The table contains **1,115 records and 10 variables**.

During the initial exploration, missing values were found mainly in variables related to **competition** and the **Promo2** program. The absence of `CompetitionOpenSinceMonth` and `CompetitionOpenSinceYear` is associated with stores that do not have information about the opening date of the nearest competitor. Similarly, `Promo2SinceWeek`, `Promo2SinceYear`, and `PromoInterval` have missing values for stores that do not participate in the Promo2 program.

Data Dictionary

| Variable                  | Description                                          | Type       | Notes                                                      |
|---------------------------|------------------------------------------------------|------------|------------------------------------------------------------|
| `Store`                   | Store identifier                                     | Integer    | Table key and used for joining with the `train` table      |
| `StoreType`               | Type of store                                        | Categorical| Classification among store types A, B, C, and D            |
| `Assortment`              | Assortment type                                      | Categorical| Classification of product variety level for the store      |
| `CompetitionDistance`     | Distance to the nearest competitor                   | Numeric    | About 26.9% of records have missing values                 |
| `CompetitionOpenSinceMonth`| Month when the nearest competitor opened            | Numeric    | About 31.7% of records have missing values                 |
| `CompetitionOpenSinceYear`| Year when the nearest competitor opened              | Numeric    | About 31.7% of records have missing values                 |
| `Promo2`                  | Participation in Promo2 program indicator            | Integer    | 1 = participates; 0 = does not participate                 |
| `Promo2SinceWeek`         | Week when the store began participating in Promo2    | Numeric    | About 48.8% of records have missing values                 |
| `Promo2SinceYear`         | Year when the store began participating in Promo2    | Numeric    | About 48.8% of records have missing values                 |
| `PromoInterval`           | Months when Promo2 is active                         | Categorical| About 48.8% of records have missing values                 |

Observed Points

* The table contains **1,115 rows and 10 columns**, with a granularity of one observation per store.
* `Store` is fully populated (100%) and will serve as the foreign key to join the `store` table with the `train` table.
* `StoreType` and `Assortment` have no missing values and describe structural characteristics of the stores.
* `CompetitionDistance` has about **26.9% missing values**, indicating some stores have no available information on the proximity of a competitor.
* `CompetitionOpenSinceMonth` and `CompetitionOpenSinceYear` have about **31.7% missing values**, related to the lack of information on when the competition opened.
* The variables `Promo2SinceWeek`, `Promo2SinceYear`, and `PromoInterval` have about **48.8% missing values**, mainly because not all stores participate in Promo2.
* Competition and Promo2 information will later be used to create temporal features, such as **time since the beginning of competition** and **time since Promo2 started**, as well as activity indicators for these events.
* The handling of missing values should consider the meaning of each variable—do not assume all missing values represent the same behavior.


In [ ]:
# Connect to the SQLite database
conn = sqlite3.connect('../../../data/database.db')

# Check available tables
tables = pd.read_sql_query(
    "SELECT name FROM sqlite_master WHERE type='table';",
    conn
)
print("Available tables in the database:", tables['name'].tolist())

# Read the 'store' table
df_store = pd.read_sql_query("SELECT * FROM store", conn)

conn.close()  # Closing the connection is important

Tabelas disponíveis no banco de dados: ['store', 'sales']


In [ ]:
# Display the first few rows of the store DataFrame
df_store.head()

,Store,StoreType,Assortment,CompetitionDistance,CompetitionOpenSinceMonth,CompetitionOpenSinceYear,Promo2,Promo2SinceWeek,Promo2SinceYear,PromoInterval
0,1,c,a,1270.0,9.0,2008.0,0,NaN,NaN,None
1,2,a,a,570.0,11.0,2007.0,1,13.0,2010.0,"Jan,Apr,Jul,Oct"
2,3,a,a,14130.0,12.0,2006.0,1,14.0,2011.0,"Jan,Apr,Jul,Oct"
3,4,c,c,620.0,9.0,2009.0,0,NaN,NaN,None
4,5,a,a,29910.0,4.0,2015.0,0,NaN,NaN,None


In [ ]:
# Display the number of rows and columns in the store DataFrame
num_rows = df_store.shape[0]
num_columns = df_store.shape[1]

print(f"Number of rows: {num_rows}")
print(f"Number of columns: {num_columns}")

Número de linhas: 1115
Número de colunas: 10


In [ ]:
# Data types of the columns in the store table
df_store.dtypes

Store                          int64
StoreType                     object
Assortment                    object
CompetitionDistance          float64
CompetitionOpenSinceMonth    float64
CompetitionOpenSinceYear     float64
Promo2                         int64
Promo2SinceWeek              float64
Promo2SinceYear              float64
PromoInterval                 object
dtype: object

In [ ]:
# Calculate the percentage of missing values per column
missing_percent = df_store.isnull().mean() * 100

print("Percentage of missing values per column:")
print(missing_percent)

Porcentagem de valores ausentes por coluna:
Store                         0.000000
StoreType                     0.000000
Assortment                    0.000000
CompetitionDistance           0.269058
CompetitionOpenSinceMonth    31.748879
CompetitionOpenSinceYear     31.748879
Promo2                        0.000000
Promo2SinceWeek              48.789238
Promo2SinceYear              48.789238
PromoInterval                48.789238
dtype: float64
